In [ ]:
from pathlib import Path
EXPECTED_GIT_COMMIT = "42c76c229028988de59cbe95c701348cfecc5824"
assert (len(EXPECTED_GIT_COMMIT) == 40 and EXPECTED_GIT_COMMIT != "REPLACE_AFTER_PUSH"), "Pin the reviewed pushed commit before Colab validation"
ACCOUNT_LABEL = ""  # A, B, or C
RUN_MODE = "fresh"  # fresh or resume; account switches use resume
STALE_MARKER_CONFIRMATION = ""  # CLEAR STALE MARKER only after the old runtime is stopped
assert ACCOUNT_LABEL in {"A", "B", "C"} and RUN_MODE in {"fresh", "resume"}
REPO_URL = "https://github.com/sfczaa/ddpm-derm-augmentation.git"
RUN_VERSION = "v4_focal_inverse_frequency_safe_v2"
CLASS_WEIGHTING = "inverse_frequency"
LOSS_NAME = "focal_cross_entropy"
FOCAL_GAMMA = 2.0
CHECKPOINT_FORMAT = "frozen_backbone_head_only_v1"
SHARED_PROJECT_DIR = Path("/content/drive/MyDrive/ddpm-derm-augmentation")
SHARED_RUN_ROOT = Path("/content/drive/MyDrive/ddpm-derm-coca-runs")
V1_ROOT = SHARED_RUN_ROOT / "sqrt_balanced_seed0_v1" / "coca_classifier" / "v1"
V2_ROOT = SHARED_RUN_ROOT / "sqrt_balanced_seed0_v1" / "coca_classifier" / "v2_weighted_ce"
V2_FAILURE_RUN = V2_ROOT / "validation_runs" / "20260720T085738Z"
V3_ROOT = SHARED_RUN_ROOT / "sqrt_balanced_seed0_v1" / "coca_classifier" / "v3_inverse_frequency_ce"
V3_LATEST_FAILURE_RECORD = V3_ROOT / "latest_validation_failure.json"
V3_VALIDATION_RECORD = V3_ROOT / "validation_record.json"
V4_ROOT = SHARED_RUN_ROOT / "sqrt_balanced_seed0_v1" / "coca_classifier" / RUN_VERSION
VALIDATION_RECORD = V4_ROOT / "validation_record.json"
LATEST_FAILURE_RECORD = V4_ROOT / "latest_validation_failure.json"
FORMAL_ROOT = V4_ROOT / "formal"
CHECKPOINT_ROOT = FORMAL_ROOT / "checkpoints" / "coca_vit_b32"
RESULTS_ROOT = FORMAL_ROOT / "results" / "coca_vit_b32"
RECORDS_ROOT = FORMAL_ROOT / "records"
EXECUTED_NOTEBOOKS_ROOT = FORMAL_ROOT / "executed_notebooks"
RUNNING_MARKER = FORMAL_ROOT / "_RUNNING.json"
COMPLETED_MARKER = FORMAL_ROOT / "_COMPLETED.json"
FORMAL_IDENTITY_RECORD = RECORDS_ROOT / "formal_identity.json"
HANDOFF_HISTORY_RECORD = RECORDS_ROOT / "handoff_history.json"
SHARED_ROOT_SENTINEL = SHARED_RUN_ROOT / ".coca_shared_root.json"
CANDIDATE_MANIFEST = SHARED_PROJECT_DIR / "outputs" / "exploratory_balanced_ddpm" / "sqrt_balanced_seed0_v1" / "candidate_synthetic_df" / "epoch0100_seed0" / "synthetic_df.csv"
EXPECTED_CANDIDATE_SHA256 = "9ef9b44e404f74aab8211f4e7d123da3258ba8ba4e3004a4147d1761ed343b34"

# CoCa v4 focal inverse-frequency C1/C4 formal runs


## Phase 0: Code, validation gate, data, and v1-v3 guards


In [ ]:
import hashlib, json, os, base64, shutil, subprocess, sys
from google.colab import drive, userdata
drive.mount("/content/drive")
assert SHARED_PROJECT_DIR.is_dir() and SHARED_RUN_ROOT.is_dir()
assert V1_ROOT.is_dir(), f"v1 must remain present and read-only: {V1_ROOT}"
assert V2_ROOT.is_dir(), f"v2 must remain present and read-only: {V2_ROOT}"
assert V3_ROOT.is_dir(), f"v3 must remain present and read-only: {V3_ROOT}"
assert V3_LATEST_FAILURE_RECORD.is_file() and not V3_VALIDATION_RECORD.exists()
assert SHARED_ROOT_SENTINEL.is_file()
if LATEST_FAILURE_RECORD.exists():
    raise RuntimeError(f"a v4 failure record exists; formal training is blocked: {LATEST_FAILURE_RECORD}")
assert VALIDATION_RECORD.is_file(), f"formal training requires a PASSED v4 validation record: {VALIDATION_RECORD}"
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""): digest.update(chunk)
    return digest.hexdigest()
v1_guard_paths = [SHARED_ROOT_SENTINEL, V1_ROOT / "validation_record.json", V1_ROOT / "formal" / "_COMPLETED.json"]
v2_failure_gate_results = V2_FAILURE_RUN / "non_collapse_gate" / "results" / "coca_vit_b32"
v2_guard_paths = [v2_failure_gate_results / "results_C1_seed0.json", v2_failure_gate_results / "results_C4_seed0.json"]
v3_guard_paths = [V3_LATEST_FAILURE_RECORD]
guard_paths = v1_guard_paths + v2_guard_paths + v3_guard_paths
assert all(path.is_file() for path in guard_paths)
before_guard = {str(path): (sha256(path), path.stat().st_mtime_ns) for path in guard_paths}
subprocess.run(["nvidia-smi"], check=True)
token = userdata.get("GH_TOKEN")
assert token and len(token) > 20, "Colab Secret GH_TOKEN with read access to this repo is required"
GH_TOKEN_PRESENT = True
CODE_DIR = Path("/content/ddpm-coca-v4-code")
assert not CODE_DIR.exists(), f"fresh runtime required: {CODE_DIR}"
basic_credential = base64.b64encode(("x-access-token:" + token).encode()).decode()
clone_env = os.environ.copy()
clone_env["GIT_CONFIG_COUNT"] = "1"
clone_env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
clone_env["GIT_CONFIG_VALUE_0"] = "Authorization: Basic " + basic_credential
try:
    subprocess.run(["git", "clone", REPO_URL, str(CODE_DIR)], check=True, env=clone_env)
finally:
    clone_env["GIT_CONFIG_VALUE_0"] = ""
    token = basic_credential = None
    del token, basic_credential, clone_env
subprocess.run(["git", "-C", str(CODE_DIR), "checkout", "--detach", EXPECTED_GIT_COMMIT], check=True)
commit = subprocess.check_output(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"], text=True).strip()
remote = subprocess.check_output(["git", "-C", str(CODE_DIR), "remote", "get-url", "origin"], text=True).strip()
status = subprocess.check_output(["git", "-C", str(CODE_DIR), "status", "--short"], text=True).strip()
assert commit == EXPECTED_GIT_COMMIT and not status, "clone must be a clean detached checkout of the pinned commit"
assert "@" not in remote and "x-access-token" not in remote, "clone URL must not embed a credential"
os.environ["HF_HOME"] = "/content/hf-cache"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch==3.3.0", "pandas>=2.0", "pillow>=9.0"], check=True)
sys.path.insert(0, str(CODE_DIR / "src"))
from importlib.metadata import version
import torch
from ddpm_derm import coca_run
assert torch.cuda.is_available() and version("open_clip_torch") == "3.3.0"
resolved_root = coca_run.require_existing_shared_root(SHARED_RUN_ROOT)
drive_probe = coca_run.probe_shared_drive(resolved_root)
sentinel = json.loads(SHARED_ROOT_SENTINEL.read_text(encoding="utf-8"))
assert sentinel["resolved_path"] == str(resolved_root) and sentinel["run_version"] == "v1"
validation = json.loads(VALIDATION_RECORD.read_text(encoding="utf-8"))
assert CANDIDATE_MANIFEST.is_file() and sha256(CANDIDATE_MANIFEST) == EXPECTED_CANDIDATE_SHA256
LOCAL_DATA_DIR = Path("/content/ham10000-data")
assert not LOCAL_DATA_DIR.exists(); shutil.copytree(SHARED_PROJECT_DIR / "data", LOCAL_DATA_DIR)
os.environ["DDPM_DERM_DATA_DIR"] = str(LOCAL_DATA_DIR)
from ddpm_derm import classifier_objective, manifests
frames = {split: manifests.load_split(split) for split in ("train", "val", "test")}
assert {key: len(value) for key, value in frames.items()} == {"train": 6995, "val": 1510, "test": 1510}
assert [int((frames[split]["dx"] == "df").sum()) for split in ("train", "val", "test")] == [85, 14, 16]
for other in ("val", "test"):
    assert not set(frames["train"]["lesion_id"]) & set(frames[other]["lesion_id"])
    assert not set(frames["train"]["image_id"]) & set(frames[other]["image_id"])
fixed_split_identity = sha256(LOCAL_DATA_DIR / "manifests" / "train.csv")
formal_output_identity = f"{sentinel['shared_root_uuid']}:sqrt_balanced_seed0_v1:coca_classifier:{RUN_VERSION}:formal"
expected_validation = {"loss_name": LOSS_NAME, "focal_gamma": FOCAL_GAMMA, "git_commit": commit, "run_version": RUN_VERSION, "candidate_manifest_sha256": EXPECTED_CANDIDATE_SHA256, "fixed_split_identity": fixed_split_identity, "shared_root_uuid": sentinel["shared_root_uuid"], "formal_output_identity": formal_output_identity, "checkpoint_format": CHECKPOINT_FORMAT, "evaluation_scope": "validation_only"}
coca_run.require_validation_record(validation, expected_validation)
assert validation["loss_name"] == LOSS_NAME and validation["focal_gamma"] == FOCAL_GAMMA
assert validation["focal_reduction"] == "weighted_mean_by_target_alpha"
assert validation["class_weighting"] == "inverse_train_frequency"
assert set(validation["non_collapse_gate"]) == {"C1", "C4"}
for gate in validation["non_collapse_gate"].values(): assert all(gate["checks"].values())
c1_full = manifests.build_classifier_frame("C1", df_target_count=585, seed=0)
objective, weights = classifier_objective.build_training_objective("inverse_frequency", c1_full, loss_name=LOSS_NAME, focal_gamma=FOCAL_GAMMA)
assert objective == validation["training_objective"]

In [ ]:
from PIL import Image
from ddpm_derm.model import build_model, model_identity, parameter_counts
sanity_model = build_model(arch="coca_vit_b32", freeze_backbone=True, coca_pretrained="laion2b_s13b_b90k").cuda()
train_frame = manifests.load_split("train")
sample_path = LOCAL_DATA_DIR / train_frame.iloc[0]["image_path"]
sample = sanity_model.eval_preprocess(Image.open(sample_path).convert("RGB")).unsqueeze(0).cuda()
sanity_model.train(); sanity_logits = sanity_model(sample)
assert tuple(sanity_logits.shape) == (1, 7) and not sanity_model.encoder.training
total_params, trainable_params = parameter_counts(sanity_model)
assert trainable_params == 3591 and total_params == 253563656
current_model_identity = model_identity(sanity_model, "coca_vit_b32", 128)
assert current_model_identity == validation["model_identity"]
del sanity_model, sanity_logits, sample; torch.cuda.empty_cache()

## Phase 1: Isolated v4 resume and run marker


In [ ]:
if RUNNING_MARKER.exists():
    print(RUNNING_MARKER.read_text(encoding="utf-8"))
    if STALE_MARKER_CONFIRMATION != "CLEAR STALE MARKER": raise RuntimeError("existing marker retained; confirm the old runtime is stopped")
    coca_run.clear_stale_marker(RUNNING_MARKER, STALE_MARKER_CONFIRMATION)
if RUN_MODE == "fresh":
    existing = [] if not FORMAL_ROOT.exists() else [path for path in FORMAL_ROOT.rglob("*") if path.is_file()]
    assert not existing, f"fresh mode refuses existing v4 formal artifacts: {existing[:10]}"
else:
    assert FORMAL_ROOT.is_dir(), f"resume requires existing v4 formal root: {FORMAL_ROOT}"
for path in (FORMAL_ROOT, CHECKPOINT_ROOT, RESULTS_ROOT, RECORDS_ROOT, EXECUTED_NOTEBOOKS_ROOT): coca_run.ensure_tree(SHARED_RUN_ROOT, path.relative_to(SHARED_RUN_ROOT))
formal_identity = {**expected_validation, "evaluation_scope": "full", "model_identity": current_model_identity, "training_objective": objective}
if RUN_MODE == "fresh": coca_run.write_json_atomic(FORMAL_IDENTITY_RECORD, formal_identity)
else:
    assert FORMAL_IDENTITY_RECORD.is_file()
    coca_run.require_resume_identity(json.loads(FORMAL_IDENTITY_RECORD.read_text(encoding="utf-8")), formal_identity)
marker = coca_run.session_marker(ACCOUNT_LABEL, RUN_MODE, {**formal_identity, "current_variant": None, "current_seed": None, "current_epoch": 0, "resolved_shared_root": str(resolved_root)})
coca_run.create_running_marker(RUNNING_MARKER, marker)
handoff = json.loads(HANDOFF_HISTORY_RECORD.read_text(encoding="utf-8"))["sessions"] if HANDOFF_HISTORY_RECORD.is_file() else []
handoff.append({"account_label": ACCOUNT_LABEL, "session_id": marker["session_id"], "hostname": marker["hostname"], "run_mode": RUN_MODE, "started_utc": marker["started_utc"]})
coca_run.write_json_atomic(HANDOFF_HISTORY_RECORD, {"sessions": handoff})
run_queue = [(variant, seed) for variant in ("C1", "C4") for seed in (0, 1, 2)]
print("fixed queue:", run_queue)
print("checkpoint cadence: every completed epoch; worst-case loss: one unfinished epoch")

## Phase 2: Six inverse-frequency 20-epoch runs


In [ ]:
from ddpm_derm.checkpoint import load_checkpoint

import re
env = os.environ.copy(); env["PYTHONPATH"] = str(CODE_DIR / "src"); env["PYTHONUNBUFFERED"] = "1"
def paths_for(variant, seed):
    root = CHECKPOINT_ROOT / f"{variant}_seed{seed}"
    return root / "best.pt", root / "last.pt", RESULTS_ROOT / f"results_{variant}_seed{seed}.json"
def validate_completed(variant, seed):
    best, last, result_path = paths_for(variant, seed)
    assert best.is_file() and last.is_file() and result_path.is_file()
    result = json.loads(result_path.read_text(encoding="utf-8"))
    coca_run.require_resume_identity(result["run_identity"], formal_identity)
    assert result["variant"] == variant and result["seed"] == seed and len(result["history"]) == 20
    assert result["evaluation_scope"] == "full" and result["test_metrics"] is not None
    assert result["run_identity"]["run_version"] == RUN_VERSION and result["run_identity"]["training_objective"] == objective
    assert result["training_objective"] == objective == result["run_identity"]["training_objective"]
    assert objective["loss_name"] == LOSS_NAME and objective["focal_gamma"] == FOCAL_GAMMA
    assert objective["focal_reduction"] == "weighted_mean_by_target_alpha"
    assert result["run_identity"]["training_objective"]["class_weighting"] == "inverse_train_frequency"
    assert result["run_identity"]["model_identity"] == current_model_identity
    assert result["run_identity"]["candidate_manifest_sha256"] == EXPECTED_CANDIDATE_SHA256
    assert result["data_counts"] == {"train": 7495, "val": 1510, "test": 1510}
    for checkpoint_path in (best, last):
        checkpoint = load_checkpoint(checkpoint_path, map_location="cpu")
        coca_run.require_resume_identity(checkpoint["run_identity"], formal_identity)
        assert checkpoint["run_identity"] == result["run_identity"] and checkpoint["checkpoint_format"] == CHECKPOINT_FORMAT
        assert "head_state_dict" in checkpoint and "model_state_dict" not in checkpoint and "encoder_state_dict" not in checkpoint
        coca_run.checkpoint_size(checkpoint_path, arch="coca_vit_b32")
        assert len(checkpoint["history"]) == checkpoint["epoch"] and 1 <= checkpoint["epoch"] <= 20
    assert load_checkpoint(last, map_location="cpu")["epoch"] == 20
    return result
def update_marker(**updates):
    state = json.loads(RUNNING_MARKER.read_text(encoding="utf-8")); state.update(updates, last_updated_utc=coca_run.utc_now()); coca_run.write_json_atomic(RUNNING_MARKER, state)
for variant, seed in run_queue:
    best, last, result_path = paths_for(variant, seed)
    if result_path.exists(): validate_completed(variant, seed); print(f"[{variant} seed {seed}] completed identity verified; skip"); continue
    update_marker(current_variant=variant, current_seed=seed, current_epoch=0)
    command = [sys.executable, "-u", "-m", "ddpm_derm.train_classifier", "--arch", "coca_vit_b32", "--freeze-backbone", "--coca-pretrained", "laion2b_s13b_b90k", "--variant", variant, "--seed", str(seed), "--epochs", "20", "--batch-size", "32", "--lr", "3e-4", "--weight-decay", "1e-4", "--df-target-count", "585", "--num-workers", "2", "--loss-name", "focal_cross_entropy", "--class-weighting", "inverse_frequency", "--focal-gamma", "2.0", "--evaluation-scope", "full", "--output-dir", str(FORMAL_ROOT), "--run-label", "coca_v4_focal_formal", "--run-version", RUN_VERSION, "--shared-root-uuid", sentinel["shared_root_uuid"], "--formal-output-identity", formal_output_identity, "--fixed-split-identity", fixed_split_identity, "--candidate-sha256", EXPECTED_CANDIDATE_SHA256]
    if variant == "C4": command += ["--generated-manifest", str(CANDIDATE_MANIFEST)]
    if RUN_MODE == "resume": command.append("--resume")
    process = subprocess.Popen(command, cwd=CODE_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
        match = re.search(r"\[epoch (\d+)/20\]", line)
        if match:
            assert "checkpoint_saved=last.pt" in line
            update_marker(current_epoch=int(match.group(1)))
    if process.wait(): raise RuntimeError(f"formal run failed: {variant} seed {seed}")
    validate_completed(variant, seed)

## Phases 3-4: Post-run check, aggregation, and records


In [ ]:
runs = [validate_completed(variant, seed) for variant, seed in run_queue]
assert len(list(RESULTS_ROOT.glob("results_*.json"))) == 6
aggregate = coca_run.aggregate_results(runs); assert aggregate["ddof"] == 0
coca_run.write_json_atomic(RECORDS_ROOT / "aggregate_results.json", aggregate)
checkpoint_paths = {f"{variant}_seed{seed}": {"best": str(paths_for(variant, seed)[0]), "last": str(paths_for(variant, seed)[1]), "result": str(paths_for(variant, seed)[2]), "best_pt_bytes": paths_for(variant, seed)[0].stat().st_size, "last_pt_bytes": paths_for(variant, seed)[1].stat().st_size} for variant, seed in run_queue}
training_record = {"status": "completed", "run_version": RUN_VERSION, "loss_name": LOSS_NAME, "focal_gamma": FOCAL_GAMMA, "focal_formula": objective["focal_formula"], "focal_reduction": objective["focal_reduction"], "class_weighting": "inverse_train_frequency", "runs": [{"variant": variant, "seed": seed} for variant, seed in run_queue], "artifact_paths": checkpoint_paths, "shared_root_identity": sentinel, "git_commit": commit, "dependency_versions": {"open_clip_torch": version("open_clip_torch"), "torch": torch.__version__}, "model_identity": current_model_identity, "training_objective": objective, "evaluation_scope": "full", "checkpoint_format": CHECKPOINT_FORMAT, "encoder_weights_stored": False, "candidate_manifest_sha256": EXPECTED_CANDIDATE_SHA256, "aggregate_results": aggregate, "validation_record": str(VALIDATION_RECORD), "start_utc": json.loads(RUNNING_MARKER.read_text(encoding="utf-8"))["started_utc"], "end_utc": coca_run.utc_now(), "account_session_handoff_history": json.loads(HANDOFF_HISTORY_RECORD.read_text(encoding="utf-8"))["sessions"], "executed_notebook_archive_path": str(EXECUTED_NOTEBOOKS_ROOT / "colab_coca_v4_focal_inverse_frequency_classifier_executed.ipynb")}
guard_after = {str(path): (sha256(path), path.stat().st_mtime_ns) for path in guard_paths}
assert guard_after == before_guard, "v1/v2/v3 read-only guard files changed"
coca_run.write_json_atomic(RECORDS_ROOT / "training_record.json", training_record)
coca_run.write_json_atomic(COMPLETED_MARKER, training_record)
RUNNING_MARKER.unlink()
assert COMPLETED_MARKER.is_file() and not RUNNING_MARKER.exists()
print(json.dumps(aggregate, indent=2))
print("FORMAL COCA V4 FOCAL RUNS COMPLETED AND VERIFIED")
print("Archive the executed notebook at:", training_record["executed_notebook_archive_path"])